# Gold - Dimensão Produtos

Consolidação das informações dos produtos com chaves substitutas de negócio.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'dim_produtos'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, coalesce, lit

df_prod = spark.read.table(f"{var_environment}.{var_silver_schema}.case_cadastro_produtos")

df_gold = (
    df_prod
    .withColumn("sk_produto", sha2(col("id_produto"), 256))
    .select(
        col("sk_produto").cast("string").alias("sk_produto"),
        col("id_produto").cast("string").alias("id_produto"),
        coalesce(col("nome_produto"), lit("Produto Desconhecido")).cast("string").alias("nome_produto"),
        col("categoria_produto").cast("string").alias("categoria_produto"),
        col("subcategoria_produto").cast("string").alias("subcategoria_produto"),
        col("status_produto").cast("string").alias("status_produto"),
        col("preco_tabela").alias("preco_tabela"),
        col("moeda").cast("string").alias("moeda"),
        col("familia_produto").cast("string").alias("familia_produto")
    )
)

In [ ]:
process_data(
    df_write=df_gold,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['categoria_produto'],
    chave_upsert='sk_produto'
)